# XR Client Visuals

Examples of dynamically creating simple visuals in the XR client from python code.

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: xr client visuals")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Length mode

In [3]:
from ipywidgets import Output
output = Output()
output

Output()

In [4]:
import numpy as np
from nanover.trajectory import FrameData
from nanover.jupyter import ImdAgent
from threading import RLock


class LengthVisuals(ImdAgent):
    _pairs: list[tuple[int, int]] = []
    _lock = RLock()

    def clear(self):
        utilities.objects.clear()
        with self._lock:
            self._pairs.clear()

    def add_pair(self, atom_a: int, atom_b: int):
        with self._lock:
            self._pairs.append((atom_a, atom_b))

    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects as objects:
            with self._lock:
                for i, (atom_a, atom_b) in enumerate(self._pairs):
                    pos_a = full_frame.particle_positions[atom_a]
                    pos_b = full_frame.particle_positions[atom_b]
                    length = np.linalg.norm(pos_a - pos_b)
                    centroid = (pos_a + pos_b) / 2

                    objects.update_line(i, positions=(pos_a, pos_b), size=0.01, color=[.5, .5, .5, .5])
                    objects.update_label(i, position=centroid, text=f"{length:.2f}nm")

length_visuals = LengthVisuals.from_runner(imd_runner)
length_visuals.start()

imd_runner.app_server.register_command("user/lengths/clear", length_visuals.clear)

In [5]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode


class LengthMode(Mode):
    prev_atom = None

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        next_atom = int(interaction.particles[0])

        if self.prev_atom is None:
            self.prev_atom = next_atom
            utilities.notify_all(f"pick next atom")
        else:
            length_visuals.add_pair(next_atom, self.prev_atom)
            utilities.notify_all(f"tracking length")
            self.prev_atom = None


utilities.use_interaction_modes()
utilities.add_interaction_mode(LengthMode, "length")

In [6]:
length_visuals._pairs

[]